In [8]:
import os
os.environ["GRB_LICENSE_FILE"] = os.path.expanduser("~/gurobi.lic")

In [9]:
import gurobipy as gp
from gurobipy import GRB
import math
import numpy as np

def solve_cvrp_gurobi(instance_name, nodes, N_v, C):
    """
    Solves the CVRP using Gurobi.
    nodes: list of (x,y) tuples. Index 0 must be the depot.
    """
    n = len(nodes)
    
    # 1. Calculate Euclidean Distance Matrix
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i,j] = math.sqrt((nodes[i][0] - nodes[j][0])**2 + (nodes[i][1] - nodes[j][1])**2)
            
    # 2. Initialize Gurobi Model
    m = gp.Model(instance_name)
    m.setParam('OutputFlag', 1) # Suppresses the long Gurobi log output
    
    # 3. Decision Variables
    # x[i,j] = 1 if a vehicle travels directly from node i to node j
    x = m.addVars(n, n, vtype=GRB.BINARY, name="x")
    
    # u[i] = continuous variable to track the "load" (customers visited) of a vehicle at node i
    u = m.addVars(n, vtype=GRB.CONTINUOUS, lb=1, ub=C, name="u")
    
    # 4. Objective: Minimize total distance
    m.setObjective(gp.quicksum(dist[i,j] * x[i,j] for i in range(n) for j in range(n)), GRB.MINIMIZE)
    
    # 5. Constraints
    m.addConstrs((x[i,i] == 0 for i in range(n)), "no_self_loops") # Cannot travel to itself
    
    # Every customer is entered exactly once and exited exactly once
    m.addConstrs((gp.quicksum(x[i,j] for i in range(n)) == 1 for j in range(1, n)), "enter_once")
    m.addConstrs((gp.quicksum(x[i,j] for j in range(n)) == 1 for i in range(1, n)), "exit_once")
    
    # Depot constraints: Use AT MOST N_v vehicles
    m.addConstr(gp.quicksum(x[0,j] for j in range(1, n)) <= N_v, "depot_out")
    m.addConstr(gp.quicksum(x[i,0] for i in range(1, n)) <= N_v, "depot_in")
    
    # Flow balance at depot (vehicles leaving must equal vehicles returning)
    m.addConstr(gp.quicksum(x[0,j] for j in range(1, n)) == gp.quicksum(x[i,0] for i in range(1, n)), "depot_bal")
    
    # MTZ Sub-tour Elimination and Capacity Constraint
    # This prevents disconnected loops and enforces that no route visits more than C customers
    for i in range(1, n):
        for j in range(1, n):
            if i != j:
                m.addConstr(u[i] - u[j] + C * x[i,j] <= C - 1, f"mtz_{i}_{j}")
                
    # 6. Solve
    m.optimize()
    
    # 7. Extract and Format Results
    if m.Status == GRB.OPTIMAL:
        print(f"--- Optimal Solution for {instance_name} ---")
        print(f"Total Distance: {m.ObjVal:.2f}")
        
        # Find active edges
        active_edges = [(i, j) for i in range(n) for j in range(n) if x[i,j].x > 0.5]
        
        # Trace routes starting from depot
        starts = [j for (i, j) in active_edges if i == 0]
        for route_idx, start_node in enumerate(starts):
            route = [0, start_node]
            current = start_node
            while True:
                next_node = [j for (i, j) in active_edges if i == current][0]
                route.append(next_node)
                current = next_node
                if current == 0: # Returned to depot
                    break
            
            # Format to match Hackathon required output
            route_str = ", ".join(map(str, route))
            print(f"r{route_idx + 1}: {route_str}")
    else:
        print(f"No optimal solution found for {instance_name}.")

In [10]:
import time

def solve_cvrp_gurobi_lazy(instance_name, nodes, N_v, C):
    """Solves CVRP using lazy subtour elimination + capacity cuts."""
    t0 = time.time()
    n = len(nodes)

    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i,j] = math.sqrt((nodes[i][0]-nodes[j][0])**2 + (nodes[i][1]-nodes[j][1])**2)

    m = gp.Model(str(instance_name))
    m.setParam('OutputFlag', 1)
    m.setParam('LazyConstraints', 1)
    m.setParam('TimeLimit', 1800) # Ensure it doesn't hang forever

    x = m.addVars(n, n, vtype=GRB.BINARY, name="x")
    m.setObjective(gp.quicksum(dist[i,j]*x[i,j] for i in range(n) for j in range(n)), GRB.MINIMIZE)

    m.addConstrs((x[i,i] == 0 for i in range(n)))
    m.addConstrs((gp.quicksum(x[i,j] for i in range(n)) == 1 for j in range(1,n)))
    m.addConstrs((gp.quicksum(x[i,j] for j in range(n)) == 1 for i in range(1,n)))
    m.addConstr(gp.quicksum(x[0,j] for j in range(1,n)) <= N_v)
    m.addConstr(gp.quicksum(x[i,0] for i in range(1,n)) <= N_v)
    m.addConstr(gp.quicksum(x[0,j] for j in range(1,n)) == gp.quicksum(x[i,0] for i in range(1,n)))

    def subtour_callback(model, where):
        if where != GRB.Callback.MIPSOL:
            return
        vals = model.cbGetSolution(x)
        succ = {}
        for i in range(n):
            for j in range(n):
                if vals[i,j] > 0.5:
                    succ[i] = j
        starts = [j for j in range(1, n) if vals[0,j] > 0.5]
        visited = set()
        for s in starts:
            route = []
            cur = s
            while cur != 0:
                route.append(cur)
                visited.add(cur)
                cur = succ.get(cur, 0)
            if len(route) > C:
                comp = set(route)
                model.cbLazy(
                    gp.quicksum(x[i,j] for i in comp for j in range(n) if j not in comp)
                    >= math.ceil(len(comp) / C)
                )
        unvisited = set(range(1, n)) - visited
        while unvisited:
            cur = unvisited.pop()
            comp = {cur}
            while succ.get(cur) in unvisited:
                cur = succ[cur]
                comp.add(cur)
                unvisited.discard(cur)
            model.cbLazy(gp.quicksum(x[i,j] for i in comp for j in comp) <= len(comp) - 1)

    m.optimize(subtour_callback)

    elapsed = time.time() - t0
    if m.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL):
        print(f"\n--- Solution for {instance_name} (Lazy Subtour) ---")
        print(f"Total Distance: {m.ObjVal:.2f}")
        print(f"MIP Gap: {m.MIPGap*100:.2f}%")
        return m.ObjVal, elapsed
    else:
        print(f"No solution found. Status: {m.Status}")
        return float('inf'), elapsed


In [11]:
import json
import os
import csv

# ── Solve ALL instances from final_instances.json ──
with open('../final_instances.json', 'r') as f:
    all_instances = json.load(f)

results = []
for config in all_instances:
    inst_id = config['instance_id']
    
    if isinstance(config['customers'][0], dict):
        nodes = [(float(c['x']), float(c['y'])) for c in config['customers']]
    else:
        nodes = [(float(c[0]), float(c[1])) for c in config['customers']]
        
    Nv = config['Nv']
    C  = config['C']

    print(f"\n{'='*60}")
    print(f"Instance: {inst_id}  |  {len(nodes)-1} customers  |  {Nv} vehicles  |  Capacity {C}")
    print(f"{'='*60}\n")
    
    obj_val, runtime = solve_cvrp_gurobi_lazy(inst_id, nodes, N_v=Nv, C=C)
    results.append([str(inst_id).split('_')[-1], 'Gurobi Lazy Exact', round(obj_val, 2), round(obj_val, 2), round(obj_val, 2), round(runtime, 2), 'Exact deterministic MILP via Gurobi'])

os.makedirs('../Gurobi_Results', exist_ok=True)
csv_file = '../Gurobi_Results/gurobi_results.csv'
with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['instance', 'config', 'trial_1', 'avg_value', 'best_value', 'avg_time_s', 'notes'])
    writer.writerows(results)

print(f"\nSaved results to {csv_file}")



Instance: 1  |  3 customers  |  2 vehicles  |  Capacity 5

Set parameter OutputFlag to value 1
Set parameter LazyConstraints to value 1
Set parameter TimeLimit to value 1800
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M3 Pro
Thread count: 11 physical cores, 11 logical processors, using up to 11 threads

Non-default parameters:
TimeLimit  1800
LazyConstraints  1

Academic license 2803510 - for non-commercial use only - registered to pa___@uconn.edu
Optimize a model with 13 rows, 16 columns and 40 nonzeros (Min)
Model fingerprint: 0xdd115f89
Model has 12 linear objective coefficients
Variable types: 0 continuous, 16 integer (16 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [3e+00, 9e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Presolve removed 4 rows and 4 columns
Presolve time: 0.00s
Presolved: 9 rows, 12 columns, 30 nonzeros
Variable types: 0 continuous, 12 int